# Messages

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM.

## Setup

Load and/or check for needed environmental variables

In [5]:
from dotenv import load_dotenv
import os

# Load environment variables from .env
load_dotenv()

# Access the AI_MODEL environment variable
ai_model = os.getenv("AI_MODEL")

## Human and AI Messages

Let's initialize our chat model and use it to create an agent

In [6]:
from langchain.chat_models import init_chat_model

# initialize the chat model ("openai:gpt-5-nano") or use a local model like
# "ollama:gpt-oss"
model = init_chat_model(ai_model, temperature=0)

In [7]:
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

# create your agent! Add the model object you just created, a prompt etc.
agent = create_agent(
    model=model,
    system_prompt="You are a full-stack comedian"
)

In [8]:
# create a human message
human_msg = HumanMessage("Hello, how are you?")

# and invoke the agent with it!
result = agent.invoke({"messages": [human_msg]})

In [9]:
print(result["messages"][-1].content)

Doing well — fully patched, zero bugs (that I know of). How are you doing today? Anything I can help with — or would you prefer a joke first?


In [10]:
print(type(result["messages"][-1]))

<class 'langchain_core.messages.ai.AIMessage'>


In [11]:
for msg in result["messages"]:
    print(f"{msg.type}: {msg.content}\n")

human: Hello, how are you?

ai: Doing well — fully patched, zero bugs (that I know of). How are you doing today? Anything I can help with — or would you prefer a joke first?



### Altenative formats
#### Strings
There are situations where LangChain can infer the role from the context, and a simple string is enough to create a message. 

In [12]:
agent = create_agent(
    model=model,
    system_prompt="You are a terse sports poet.",  # This is a SystemMessage under the hood
)

In [13]:
result = agent.invoke({"messages": "Tell me about baseball"})   # This is a HumanMessage under the hood
print(result["messages"][-1].content)

Nine players. Nine innings. Three outs to end a whisper of offense, then the other team answers.

The pitcher and batter stage a private war: fastball, curve, slider — the crack is the referee. Three strikes and you go back to the dugout; four balls and you walk to first. Fair or foul changes the poem.

Bases sit like milestones — 90 feet apart. The rubber to the plate is 60 feet, 6 inches. Run them right, and you score a run. Hit the ball out of the park, and everyone's ledger changes.

Positions: pitcher, catcher, first, second, shortstop, third, left, center, right (plus the DH in some leagues). Defense is geometry; shifts are modern chess.

Basic plays: single, double, triple, home run, strikeout, walk, double play, stolen base, sacrifice fly. Errors and passed balls are the human accents.

Statistics track the story: batting average, on-base percentage, slugging — ERA for pitchers, and modern attempts like WAR to weigh legacy.

Seasons swell into pennants, the World Series crowns 

#### Dictionaries

In [14]:
result = agent.invoke(
    {"messages": {"role": "user", "content": "Write a haiku about sprinters"}}
)
print(result["messages"][-1].content)

Gun cracks — feet rocket
Muscle, focus, lightning stride
Time shivers, then stops


There are multiple roles:
```python
messages = [
    {"role": "system", "content": "You are a sports poetry expert who completes haikus that have been started"},
    {"role": "user", "content": "Write a haiku about sprinters"},
    {"role": "assistant", "content": "Feet don't fail me..."}
]
```

## Output Format
### messages
Let's create a tool so agent will create some tool messages. 

In [15]:
from langchain_core.tools import tool

@tool
def check_haiku_lines(text: str):
    """Check if the given haiku text has exactly 3 lines.

    Returns None if it's correct, otherwise an error message.
    """
    # Split the text into lines, ignoring leading/trailing spaces
    lines = [line.strip() for line in text.strip().splitlines() if line.strip()]
    print(f"checking haiku, it has {len(lines)} lines:\n {text}")

    if len(lines) != 3:
        return f"Incorrect! This haiku has {len(lines)} lines. A haiku must have exactly 3 lines."
    return "Correct, this haiku has 3 lines."

In [16]:
agent = create_agent(
    model=model,
    tools=[check_haiku_lines],
    system_prompt="You are a sports poet who only writes Haiku. You always check your work.",
)

In [17]:
result = agent.invoke({"messages": "Please write me a poem"})

checking haiku, it has 3 lines:
 Cleats bite autumn grass
Ball arcs into waiting hands
Cheers fold into dusk
checking haiku, it has 3 lines:
 Cleats drum wet twilight
A pass splits the hush of night
Turf remembers feet
checking haiku, it has 3 lines:
 Goal light blooms like dawn
Sweat and turf braid into cheers
Night keeps the ball's hush
checking haiku, it has 3 lines:
 Leather spins through sky
Crowd rises like ocean tide
Net cradles the day
checking haiku, it has 3 lines:
 Whistle splits the air
Cleats drum a steady thunder
Night keeps the echo
checking haiku, it has 3 lines:
 Cleats bite morning dew
A swift pass arcs through the haze
Net sings, crowd exhales
checking haiku, it has 3 lines:
 Cleats cut morning dew
A swift pass finds empty space
Net folds into light
checking haiku, it has 3 lines:
 Cleats whisper on grass
A long pass threads moonlit air
Net answers with light
checking haiku, it has 3 lines:
 Cleats cut morning grass
A cross cleaves the quiet air
Net folds into roar
c

KeyboardInterrupt: 

In [18]:
result["messages"][-1].content

'Gun cracks — feet rocket\nMuscle, focus, lightning stride\nTime shivers, then stops'

In [19]:
print(len(result["messages"]))

2


In [20]:
for i, msg in enumerate(result["messages"]):
    msg.pretty_print()

================================ Human Message =================================

Write a haiku about sprinters
================================== Ai Message ==================================

Gun cracks — feet rocket
Muscle, focus, lightning stride
Time shivers, then stops


### Other useful information
Above, the print messages have just been selecting pieces of the information stored in the messages list. Let's dig into all the information that is available!

In [21]:
result

{'messages': [HumanMessage(content='Write a haiku about sprinters', additional_kwargs={}, response_metadata={}, id='434989cc-2af7-4c67-a786-17674e316a48'),
  AIMessage(content='Gun cracks — feet rocket\nMuscle, focus, lightning stride\nTime shivers, then stops', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 861, 'prompt_tokens': 25, 'total_tokens': 886, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 832, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EQEJjkrHUl7On23rzwP1hNwlF9D7j', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0bf90-dc68-7a60-bc8a-5d6770c0dc30-0', tool_calls=[], invalid_tool_ca

You can select just the last message, and you can see where the final message is coming from.

In [22]:
result["messages"][-1]

AIMessage(content='Gun cracks — feet rocket\nMuscle, focus, lightning stride\nTime shivers, then stops', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 861, 'prompt_tokens': 25, 'total_tokens': 886, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 832, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EQEJjkrHUl7On23rzwP1hNwlF9D7j', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a0bf90-dc68-7a60-bc8a-5d6770c0dc30-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 25, 'output_tokens': 861, 'total_tokens': 886, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_det

In [23]:
result["messages"][-1].usage_metadata

{'input_tokens': 25,
 'output_tokens': 861,
 'total_tokens': 886,
 'input_token_details': {'audio': 0, 'cache_read': 0},
 'output_token_details': {'audio': 0, 'reasoning': 832}}

In [24]:
result["messages"][-1].response_metadata

{'token_usage': {'completion_tokens': 861,
  'prompt_tokens': 25,
  'total_tokens': 886,
  'completion_tokens_details': {'accepted_prediction_tokens': 0,
   'audio_tokens': 0,
   'reasoning_tokens': 832,
   'rejected_prediction_tokens': 0,
   'text_tokens': None},
  'prompt_tokens_details': {'audio_tokens': 0,
   'cache_write_tokens': None,
   'cached_tokens': 0,
   'image_tokens': None,
   'text_tokens': None}},
 'model_provider': 'openai',
 'model_name': 'gpt-5-mini-2025-08-07',
 'system_fingerprint': None,
 'id': 'chatcmpl-EQEJjkrHUl7On23rzwP1hNwlF9D7j',
 'service_tier': 'default',
 'finish_reason': 'stop',
 'logprobs': None}

### Try it on your own!
Change the system prompt, use the `pretty_printer` to print some messages or dig through `results` on your own. Notice the Human, AI and Tool messages and some of their associated metadata. Notice how the final results provide a complete history of the agents activity!

In [ ]:
agent = create_agent(
    model=model,
    tools=[check_haiku_lines],
    system_prompt="Your SYSTEM prompt here",
)

In [ ]:
for i, msg in enumerate(result["messages"]):
    msg.pretty_print()